# Import modules

In [ ]:
import os
import sys
import requests                      # HTTP client for API calls
import pandas as pd                  # Tabular data handling
from datetime import datetime        # Datetime handling
from typing import Iterable, Optional, Dict, Union
import matplotlib.pyplot as plt
import yfinance as yf
from pprint import pprint as pp

# Ensure repo root is on PYTHONPATH (CI safety)

In [ ]:
REPO_ROOT = os.path.abspath(os.getcwd())
SRC_PATH = os.path.join(REPO_ROOT, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)


# Secrets validation (FAIL FAST)

In [ ]:
REQUIRED_ENV_VARS = [
    "EMAIL_USER",
    "EMAIL_SENDER",
    "EMAIL_SENDER_PSW",
]

missing = [v for v in REQUIRED_ENV_VARS if not os.getenv(v)]
if missing:
    raise RuntimeError(
        f"Missing required environment variables: {', '.join(missing)}"
    )

email_user = os.getenv("EMAIL_USER")
email_sender_psw = os.getenv("EMAIL_SENDER_PSW")
email_sender = os.getenv("EMAIL_SENDER")

# Import internal modules

In [ ]:
from src.debug_print import debug_print
from src.fetch_lse_tickers import get_ftse100
from src.exchange_rates import get_share_prices_2 as share_prices
from src.exchange_rates_v2 import get_share_prices_2_with_fundamentals
from src.roi_hit_v2 import get_first_roi_hit
from src.plot_shares_ROI import plot_candles_volatility_volume_roi as ROI
from src.extract_latest_fundamentals import extract_latest_fundamentals
from src.detect_undervalued import detect_undervalued
from src.utils.email_sender import (
    send_email_html_multi_inline_images,
    build_roi_email_with_action_images,
)

# Set up variables

In [ ]:
base_currency = "GBP"
target_currencies = ["USD", "GBP", "EUR", "JPY"]
cryptos = ["BTC", "ETH"]
shares = ['FRES','ENT','GLEN','MNG','PHNX','VOD']

start_date = pd.Timestamp(2024,1,1)
purchase_date = pd.Timestamp(2026,1,2)
end_date = pd.Timestamp.today().normalize()
ROI_target = 0.135


# Ensure output directory exists (CI requirement)

In [ ]:
pics_dir = os.path.join(os.getcwd(), "output")
os.makedirs(pics_dir, exist_ok=True)


# Get TOP 100 shares from FTSE

In [ ]:
ftse100 = get_ftse100()
ftse100["Yahoo_Ticker"] = ftse100["Ticker"] + ".L"
shares_lse = ftse100["Yahoo_Ticker"].to_list()

# SHARES PRICES WITH INFO

In [ ]:
df_shares_fund = get_share_prices_2_with_fundamentals(
    tickers=shares_lse,
    start=start_date,
    end=end_date,
    base_currency = base_currency,
    vol_window = 20,
    
)

actions_list   = df_shares_fund.columns.get_level_values("ACTION").unique().to_list()
currencies_list = df_shares_fund.columns.get_level_values("CURRENCY").unique()
metrics   = df_shares_fund.columns.get_level_values("METRIC").unique()

# EXTRACT UNDERVALUED SHARES

In [ ]:
df_fund = extract_latest_fundamentals(
    df=df_shares_fund,
    evaluation_date=purchase_date,
)

undervalued_shares = detect_undervalued(df_fund)
filt = undervalued_shares[undervalued_shares["UndervaluedScore"] > 0]
undervalued_shares_list = filt.index.to_list()

# ANALYZE SHARES' PORTFOLIO to get ROI Dates

In [ ]:
portfolio = {}
SHARES_FULL_LIST = [s + '.L_GBp→GBP' if not s.endswith('.L_GBp→GBP') else s for s in shares]

for action in SHARES_FULL_LIST:
    try:
        action_clean = action.split(".L")[0]
        portfolio[action_clean] = get_first_roi_hit(
        df=df_shares_fund,
        action=action,
        purchase_date=purchase_date,
        roi_target=ROI_target
    )
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

#  PLOT SHARE - plot_candles_volatility_volume_roi 


In [ ]:
SHARES_FULL_LIST = [
    s + ".L_GBp→GBP" if not s.endswith(".L_GBP→GBP") else s
    for s in shares
]

actions = df_shares_fund.columns.get_level_values("ACTION").unique()
mask = actions.str.contains("|".join(shares), case=False, regex=True)
filtered_actions = actions[mask].to_list()

ROI(
    df=df_shares_fund,
    actions=filtered_actions,
    start=df_shares_fund.index.min(),
    end=df_shares_fund.index.max(),
    purchase_date=purchase_date,
    roi_target=ROI_target,
)

# Build email content

In [ ]:
pics_dir = os.path.join(os.getcwd(),"output")

try:
    text_body, html_body, inline_images = build_roi_email_with_action_images(
        roi_data=portfolio, #df_fres,
        image_dir=pics_dir,  # contains AAL.png, VOD.png, ...
    )
except Exception as e:
    print(f"Could not run build_roi_email_with_action_images {type(e).__name__}: {e}")


# Send email (TLS via your utility)

In [ ]:

try:
    send_email_html_multi_inline_images(
        smtp_server="smtp.gmail.com",
        smtp_port=587,
        username=email_user,
        password=email_sender_psw,
        sender=email_sender,
        recipients=["ingcarldan@gmail.com"],
        subject=f"FTSE100 - ROI targets – {datetime.today():%d-%m-%Y %H:%M}",
        text_body=text_body,
        html_body=html_body,
        inline_images=inline_images,
    )
except Exception as e:
    print(f"Could not run send_email_html_multi_inline_images {type(e).__name__}: {e}")

# PLOT PURCHASED SHARES

In [ ]:
# check if shares name ends with .L_GBP→GBP
for i, s in enumerate(shares):
    if not s.endswith('.L_GBp→GBP'):
        shares[i] = s + '.L_GBp→GBP'

ROI(
    df=df_shares_fund,
    actions=shares,
    start=df_shares_fund.index.min(),
    end=df_shares_fund.index.max(),
    purchase_date= purchase_date,
    roi_target=ROI_target
)